# SAM 2 - Segment Anything Model 2

## Detecção de Objetos Flutuantes

O **SAM 2** (Segment Anything Model 2) é o modelo de segmentação mais recente da Meta AI, lançado em Julho de 2024.

### Características:
- **Zero-shot**: Funciona sem necessidade de treinamento adicional
- **Multi-modal**: Aceita prompts de pontos, bounding boxes, máscaras ou texto
- **Vídeo**: Primeira versão com suporte nativo a vídeos
- **Arquitetura**: Hiera Vision Transformer + Memory Attention

In [ ]:
# Instalação das dependências
!pip install opencv-python numpy matplotlib Pillow -q

In [ ]:
import os
import numpy as np
import cv2
from PIL import Image
import matplotlib.pyplot as plt

os.makedirs('output', exist_ok=True)

In [ ]:
class FloatingObjectDetector:
    """Detector de objetos flutuantes usando segmentação por cor."""
    
    def __init__(self):
        self.color_ranges = {
            "rosa": {"lower": [150, 50, 100], "upper": [180, 255, 255]},
            "ciano": {"lower": [80, 50, 100], "upper": [100, 255, 255]},
            "magenta": {"lower": [130, 50, 100], "upper": [160, 255, 255]},
            "azul": {"lower": [100, 50, 100], "upper": [130, 255, 255]},
            "roxo": {"lower": [120, 50, 100], "upper": [145, 255, 255]},
        }
    
    def detect(self, image):
        if isinstance(image, str):
            img = cv2.imread(image)
        else:
            img = cv2.cvtColor(np.array(image), cv2.COLOR_RGB2BGR)
        
        hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
        result = img.copy()
        objects = []
        
        colors = [(255, 0, 255), (0, 255, 255), (255, 0, 128), (0, 128, 255), (128, 0, 255)]
        
        for i, (color_name, ranges) in enumerate(self.color_ranges.items()):
            lower = np.array(ranges["lower"])
            upper = np.array(ranges["upper"])
            
            mask = cv2.inRange(hsv, lower, upper)
            kernel = np.ones((5, 5), np.uint8)
            mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)
            
            contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
            
            for contour in contours:
                area = cv2.contourArea(contour)
                if area > 800:
                    x, y, w, h = cv2.boundingRect(contour)
                    cv2.rectangle(result, (x, y), (x + w, y + h), colors[i % len(colors)], 3)
                    cv2.putText(result, color_name, (x, y - 10),
                               cv2.FONT_HERSHEY_SIMPLEX, 0.6, colors[i % len(colors)], 2)
                    objects.append({"color": color_name, "bbox": (x, y, w, h), "area": area})
        
        result_rgb = cv2.cvtColor(result, cv2.COLOR_BGR2RGB)
        return result_rgb, objects

In [ ]:
# Carregue sua imagem aqui
IMAGE_PATH = "images/cyberpunk_image.png"  # Altere para o caminho da sua imagem

if os.path.exists(IMAGE_PATH):
    detector = FloatingObjectDetector()
    result_image, objects = detector.detect(IMAGE_PATH)
    
    plt.figure(figsize=(15, 10))
    plt.imshow(result_image)
    plt.title(f'Objetos Flutuantes Detectados: {len(objects)}')
    plt.axis('off')
    plt.show()
    
    print(f"\nTotal de objetos detectados: {len(objects)}")
    for i, obj in enumerate(objects):
        print(f"  {i+1}. {obj['color']} - Área: {obj['area']:.0f}px²")
else:
    print(f"Imagem não encontrada: {IMAGE_PATH}")
    print("Coloque sua imagem no diretório 'images/' e atualize IMAGE_PATH")

## Usando SAM 2 (Opcional)

Para usar o SAM 2 completo, instale:

In [ ]:
# !pip install git+https://github.com/facebookresearch/segment-anything-2.git
# Baixe os checkpoints de: https://github.com/facebookresearch/segment-anything-2#download-checkpoints